# Kiểm chứng  — Bản đề xuất đề tài DS317

**Đề tài:** Xây dựng mô hình phân loại nhóm nguy cơ đột quỵ dựa trên đặc điểm sức khỏe và lối sống

**Dữ liệu:** CDC BRFSS 2023 (đã xử lý — "cdc_brfss_2023_stroke_decoded.csv")

Notebook này thực hiện 3 kiểm chứng bắt buộc theo Phần 4 của hướng dẫn:
1. Tạo/đếm phân bố nhãn mục tiêu
2. Đếm số mẫu còn lại sau khi lọc
3. Chạy một baseline đơn giản để kiểm tra dữ liệu có tín hiệu dự đoán hay không

In [ ]:
import pandas as pd

df = pd.read_csv('cdc_brfss_2023_stroke_decoded.csv')
print('Kích thước dữ liệu:', df.shape)

df.head()

Kích thước dữ liệu: (190710, 18)


,age,sex,employment,bmi_category,smoking_status,heavy_drinking,physical_activity,high_bp,high_cholesterol,diabetes,heart_attack,heart_disease,asthma,kidney_disease,general_health,physical_health_days,mental_health_days,stroke
0,76,Female,Unable to work,Overweight,Never smoked,No,Yes,Yes,No,Yes,No,No,No,No,Fair,0,0,0
1,59,Male,Out of work for less than 1 year,Obese,Never smoked,No,Yes,Yes,Yes,No,No,No,No,No,Good,0,0,0
2,78,Female,Employed for wages,Normal weight,Never smoked,No,No,Yes,Yes,No,No,No,No,No,Very good,2,0,0
3,80,Female,A homemaker,Overweight,Never smoked,No,Yes,Yes,No,No,No,No,No,No,Fair,4,0,0
4,80,Female,A homemaker,Normal weight,Never smoked,No,Yes,Yes,Yes,No,Yes,Yes,No,No,Very good,0,0,0


**KIỂM CHỨNG 1: PHÂN BỐ NHÃN MỤC TIÊU STROKE**

In [ ]:
print('Phân bố nhãn stroke:')
print(df['stroke'].value_counts())
print('Tỷ lệ % từng lớp:')
print((df['stroke'].value_counts(normalize=True) * 100).round(2))
print()

Phân bố nhãn stroke:
stroke
0    185215
1      5495
Name: count, dtype: int64
Tỷ lệ % từng lớp:
stroke
0    97.12
1     2.88
Name: proportion, dtype: float64



**KIỂM CHỨNG 2: SỐ MẪU SAU KHI LỌC **

In [ ]:
print('Số dòng, số cột sau khi lọc (đã loại Don\'t know/Refused/Missing):', df.shape)
print()
print('Số giá trị thiếu theo từng cột:')
print(df.isna().sum())
print()

Số dòng, số cột sau khi lọc (đã loại Don't know/Refused/Missing): (190710, 18)

Số giá trị thiếu theo từng cột:
age                     0
sex                     0
employment              0
bmi_category            0
smoking_status          0
heavy_drinking          0
physical_activity       0
high_bp                 0
high_cholesterol        0
diabetes                0
heart_attack            0
heart_disease           0
asthma                  0
kidney_disease          0
general_health          0
physical_health_days    0
mental_health_days      0
stroke                  0
dtype: int64



**KIỂM CHỨNG 3: Chạy trên trên mô hình đơn giản để kiểm tra dữ liệu có tín hiệu không**


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix)

RANDOM_STATE = 42

X_train, X_val, X_test = load_X('X_train.csv'), load_X('X_val.csv'), load_X('X_test.csv')
y_train, y_val, y_test = load_y('y_train.csv'), load_y('y_val.csv'), load_y('y_test.csv')



cat_cols = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_cols = X_train.select_dtypes(include=['number']).columns.tolist()


# ============ MÔ HÌNH BASELINE ============
prep = ColumnTransformer([
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())]), num_cols),
])
lr_base = Pipeline([
    ('prep', prep),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
lr_base.fit(X_train, y_train)

# ============ ĐÁNH GIÁ (ngưỡng 0,5) ============
def evaluate(model, X, y, split, thr=0.5):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return {'Tập': split,
            'Accuracy': accuracy_score(y, pred),
            'Precision': precision_score(y, pred, zero_division=0),
            'Recall': recall_score(y, pred),
            'F1 (đột quỵ)': f1_score(y, pred),
            'Macro F1': f1_score(y, pred, average='macro'),
            'ROC-AUC': roc_auc_score(y, proba),
            'PR-AUC': average_precision_score(y, proba),
            'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}

results = pd.DataFrame([
    evaluate(lr_base, X_train, y_train, 'Train'),
    evaluate(lr_base, X_val,   y_val,   'Val'),
    evaluate(lr_base, X_test,  y_test,  'Test'),
]).round(3)

pd.set_option('display.width', 250, 'display.max_columns', 30)
print(results.to_string(index=False))

  Tập  Accuracy  Precision  Recall  F1 (đột quỵ)  Macro F1  ROC-AUC  PR-AUC     TN  FP   FN  TP
Train     0.971      0.378   0.024         0.045     0.515    0.845   0.178 147999 173 4291 105
  Val     0.971      0.395   0.031         0.057     0.521    0.841   0.174  18495  26  533  17
 Test     0.971      0.463   0.035         0.064     0.525    0.833   0.177  18500  22  530  19
